# Práctica de la lezione 10: clustering con K-Means

En esta pratica aplicas lo que viste en lezione sobre K-Means, tanto la versión de scikit-learn como la que programaste desde cero.

Encuentras dos esercizios. Cada uno trae un consegna con suggerimenti, una o varias cellas de código con partes marcadas como `# TODO` para que completes, y más abajo una soluzione colapsada por si te atascas. Intenta resolverlo da solo antes de mirarla.

Esta pratica no necesita ninguna dependencia extra: solo pandas, NumPy, Matplotlib y scikit-learn, que ya vienen instalados en Google Colab. Eso sí, el primer esercizio descarga el conjunto de dati del Titanic desde una URL, así que necesitas conexión a internet para ejecutarlo.

## Ejercicio 1: convierte `Embarked` a números y prueba con tres clústeres

En lezione convertiste `Sex` a números y aplicaste `KMeans` con dos clústeres sobre el Titanic. Ora tocca a te completar la conversión de otra colonna de texto, `Embarked`, que indica el puerto donde embarcó cada pasajero (`S`, `C` o `Q`, con algún hueco), y volver a agrupar a los pasajeros, esta vez en tres clústeres en lugar de dos.

**Suggerimenti:**

- `Embarked` tiene tres valorees de texto distintos. Crea un diccionario, por ejemplo `mapa_embarked`, que asigne un número entero a cada uno de esos tres valorees, igual que ya se hizo con `Sex` en la cella de abajo
- Aplica ese diccionario con `.map(mapa_embarked)` sobre `df_modello["Embarked"]`
- Después de convertir las colonnas de texto, todavía quedan huecos en colonnas numéricas como `Age` o el propio `Embarked` ya convertido; rellénalos con `df_modello.fillna(0)`
- Para el modello, usa `KMeans(n_clusters=3, n_init=10, random_state=42)`, igual que en lezione pero con `n_clusters=3`
- Como ahora hay tres grupos, ya no vale comparar "acierto directo" contra "acierto invertido" como en lezione con dos grupos; en su lugar, para cada clúster calcula cuántos pasajeros contiene y qué proporción de ellos sobrevivió, con `y[mask].mean()`

In [ ]:
import pandas as pd

# Esta celda necesita conexión a internet: descarga el CSV en el momento
# de ejecutarla. Si la descarga tarda unos segundos, es normal, solo
# espera a que termine antes de seguir con la siguiente celda.
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

df_modelo = df.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1)

# La columna Sex ya la convertimos por ti, igual que en clase
mapa_sex = {"male": 0, "female": 1}
df_modelo["Sex"] = df_modelo["Sex"].map(mapa_sex)

print(df_modelo["Embarked"].unique())

In [ ]:
# TODO 1: crea un diccionario mapa_embarked que asigne un número entero
# distinto a cada uno de los tres valores de Embarked ('S', 'C', 'Q')
mapa_embarked = ...

# TODO 2: aplica mapa_embarked sobre df_modelo["Embarked"] con .map(...)
df_modelo["Embarked"] = ...

# TODO 3: rellena con cero cualquier valor ausente que quede en df_modelo
# (piensa en Age, y también en los huecos de Embarked que no estaban
# en tu diccionario)
df_modelo = ...

print(df_modelo.dtypes)
print("Valores ausentes por columna:")
print(df_modelo.isna().sum())

Comprueba que `df_modello.dtypes` solo muestra `int64` o `float64`, y que `isna().sum()` da cero en todas las colonnas, antes de seguir. Ahora aplica K-Means con tres clústeres.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

y = df_modelo["Survived"]
X = df_modelo.drop(["Survived"], axis=1)

escalador = StandardScaler()
X_escalado = escalador.fit_transform(X)

# TODO 4: crea un KMeans con n_clusters=3, n_init=10, random_state=42
# y ajústalo (fit) sobre X_escalado
modelo = ...

etiquetas = modelo.labels_

# TODO 5: para cada uno de los tres clústeres (0, 1 y 2), calcula cuántos
# pasajeros contiene y qué proporción sobrevivió (y[mask].mean()),
# usando una máscara booleana como etiquetas == cluster
for cluster in range(3):
    mask = ...
    n_pasajeros = ...
    tasa_supervivencia = ...
    print(f"Clúster {cluster}: {n_pasajeros} pasajeros, tasa de supervivencia {tasa_supervivencia:.2f}")

<details>
<summary>💡 Vedi soluzione</summary>

```python
# TODO 1: un número entero distinto para cada uno de los tres puertos
mapa_embarked = {"S": 0, "C": 1, "Q": 2}

# TODO 2: aplicamos el diccionario sobre la colonna de texto
df_modello["Embarked"] = df_modello["Embarked"].map(mapa_embarked)

# TODO 3: rellenamos con cero cualquier hueco restante (Age, y los
# valorees de Embarked que no aparecían en el diccionario)
df_modello = df_modello.fillna(0)

print(df_modello.dtypes)
print("Valores ausentes por colonna:")
print(df_modello.isna().sum())

import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

y = df_modello["Survived"]
X = df_modello.drop(["Survived"], axis=1)

escalador = StandardScaler()
X_escalado = escalador.fit_transform(X)

# TODO 4: creamos el modello con tres clústeres y lo ajustamos
modello = KMeans(n_clusters=3, n_init=10, random_state=42)
modello.fit(X_escalado)

etiquetas = modello.labels_

# TODO 5: tamaño y tasa de supervivencia de cada uno de los tres clústeres
for cluster in range(3):
    mask = etiquetas == cluster
    n_pasajeros = mask.sum()
    tasa_supervivencia = y[mask].mean()
    print(f"Clúster {cluster}: {n_pasajeros} pasajeros, tasa de supervivencia {tasa_supervivencia:.2f}")
```

Con esta preparación de dati obtienes algo cercano a: clúster 0 con 106 pasajeros y una tasa de supervivencia de 0,40; clúster 1 con 537 pasajeros y una tasa de 0,25; clúster 2 con 248 pasajeros y una tasa de 0,66. Fíjate en que, aunque K-Means nunca vio la colonna `Survived`, un clúster concentra una tasa de supervivencia claramente más alta que los otros dos. Igual que viste en lezione con dos clústeres, subir a tres no da una traducción perfecta a "sobrevivió" o "no sobrevivió", pero sigue reflejando un patrón real: probablemente ese clúster con tasa 0,66 agrupa sobre todo a pasajeras y a pasajeros de primera lezione.

</details>

## Ejercicio 2: completa el paso de asignación y el paso de recálculo de K-Means

En lezione programaste `KMeansDesdeCero` como una lezione completa. Ahora vas a completar una versión simplificada, en forma de dos funciones sueltas, que reproducen los dos pasos centrales del algoritmo de Lloyd: la asignación de cada punto a su centroide más cercano, y el recálculo de cada centroide como el promedio de sus puntos asignados. Lo haces sobre un dataset distinto al de lezione, con cuatro grupos en lugar de tres.

**Suggerimenti:**

- En `asignar_puntos_a_centroides`, para cada punto calcula su distancia a todos los centroides con `np.linalg.norm(punto - centroide)` dentro de una lista por comprensión, y usa `np.argmin(...)` para quedarte con el índice del valore más pequeño
- En `recalcular_centroides`, selecciona primero los puntos cuya asignación coincide con el clúster actual (una máscara booleana, como en el esercizio uno) y después calcula su promedio con `np.mean(..., axis=0)`
- El bucle que llama a ambas funciones varias veces seguidas ya está escrito, no hace falta que lo toques

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

# Un dataset distinto al de clase: cuatro grupos en lugar de tres
datos, _ = make_blobs(n_samples=100, centers=4, cluster_std=1.0, random_state=7)

plt.scatter(datos[:, 0], datos[:, 1], c="lightgray", s=30, edgecolors="k")
plt.title("Dataset de práctica: cuatro grupos")
plt.show()

In [ ]:
def asignar_puntos_a_centroides(datos, centroides):
    """Devuelve una lista con el índice del centroide más cercano para
    cada punto de 'datos'."""
    asignaciones = []
    for punto in datos:
        # TODO 1: calcula la distancia de 'punto' a cada centroide de
        # 'centroides' (usa np.linalg.norm(punto - centroide) para cada uno)
        distancias = ...

        # TODO 2: obtén el índice del centroide con distancia mínima
        # (np.argmin sobre la lista de distancias) y añádelo a asignaciones
        indice_mas_cercano = ...
        asignaciones.append(indice_mas_cercano)

    return asignaciones


def recalcular_centroides(datos, asignaciones, k):
    """Devuelve un array con k centroides nuevos, cada uno el promedio
    de los puntos que tiene asignados."""
    asignaciones = np.array(asignaciones)
    nuevos_centroides = []

    for indice_cluster in range(k):
        # TODO 3: selecciona los puntos de 'datos' cuya asignación es
        # igual a indice_cluster, con una máscara booleana
        puntos_del_cluster = ...

        # TODO 4: calcula el promedio de esos puntos con np.mean(..., axis=0)
        centroide_nuevo = ...
        nuevos_centroides.append(centroide_nuevo)

    return np.array(nuevos_centroides)

In [ ]:
# Este bucle ya está completo: llama a tus dos funciones varias vueltas
# seguidas, igual que hace 'ajustar' en la clase KMeansDesdeCero de clase
k = 4
centroides = datos[:k].copy()

for iteracion in range(10):
    asignaciones = asignar_puntos_a_centroides(datos, centroides)
    centroides = recalcular_centroides(datos, asignaciones, k)

print("Centroides finales:")
print(centroides)

plt.scatter(datos[:, 0], datos[:, 1], c=asignaciones, cmap="viridis", s=30, edgecolors="k")
plt.scatter(centroides[:, 0], centroides[:, 1], c="red", marker="X", s=200, edgecolors="black")
plt.title("Resultado final de tu K-Means simplificado")
plt.show()

<details>
<summary>💡 Vedi soluzione</summary>

```python
def asignar_puntos_a_centroides(dati, centroides):
    """Devuelve una lista con el índice del centroide más cercano para
    cada punto de 'dati'."""
    asignaciones = []
    for punto in dati:
        # TODO 1: distancia euclídea del punto a cada centroide
        distancias = [np.linalg.norm(punto - centroide) for centroide in centroides]

        # TODO 2: índice del centroide más cercano
        indice_mas_cercano = int(np.argmin(distancias))
        asignaciones.append(indice_mas_cercano)

    return asignaciones


def recalcular_centroides(dati, asignaciones, k):
    """Devuelve un array con k centroides nuevos, cada uno el promedio
    de los puntos que tiene asignados."""
    asignaciones = np.array(asignaciones)
    nuevos_centroides = []

    for indice_cluster in range(k):
        # TODO 3: puntos asignados a este clúster
        puntos_del_cluster = dati[asignaciones == indice_cluster]

        # TODO 4: promedio de esos puntos, un centroide nuevo
        centroide_nuevo = np.mean(puntos_del_cluster, axis=0)
        nuevos_centroides.append(centroide_nuevo)

    return np.array(nuevos_centroides)
```

Al ejecutar el bucle con estas dos funciones completas, los centroides finales quedan muy cerca del centro real de cada uno de los cuatro grupos generados por `make_blobs`, y el gráfico final muestra cuatro colores bien diferenciados con una X roja en el centro de cada uno. Es exactamente la misma lógica de dos pasos que programaste en lezione dentro de `KMeansDesdeCero`, solo que aquí, en lugar de un metodo `ajustar` que guarda todo en `self`, cada paso vive en su propia funzione independiente.

</details>

## Resumen: lo que te llevas de esta pratica

- Convertir una colonna de texto a números con un diccionario de mapeo funciona igual sea cual sea la colonna: `Sex`, `Embarked` o cualquier otra categoría con un número limitado de valorees distintos
- Subir el número de clústeres de dos a tres no traduce automáticamente los grupos a "sobrevivió" o "no sobrevivió", pero sigue reflejando patrones reales del dataset, como la tasa de supervivencia distinta de cada grupo
- El algoritmo de Lloyd se puede escribir igual de bien como una lezione con estado interno, como hiciste en lezione, o como un par de funciones puras que reciben y devuelven dati explícitamente
- El paso de asignación siempre se reduce a lo mismo: calcular distancias a cada centroide y quedarte con la mínima; el paso de recálculo siempre es un promedio de los puntos de cada grupo